<a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSCI 447/547 Hackathon 2: Linear Regression

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the salary example together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

##### In this exercise, you will implement linear regression and get to see it work on data.

---

### Google Colab Instructions

If you are using Google Colab <a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Google Colab"/></a> you will need to download [these two files](https://github.com/lucywowen/csci547_ML/tree/main/examples/data/linear_regression_hack) and upload them to Colab for this to work.

---

### Local (VSCode/Jupyter Lab) Version

 

If you are running this locally, you will need to clone the repository locally.

First, you will need to [install Git](https://git-scm.com/install/). Follow the instructions on the website.

 

**AFTER YOU INSTALL GIT**, you will need to clone the repository, by opening powershell or bash and typing:

```bash
git clone https://github.com/lucywowen/csci547_ML.git
```

 

Now you will need to get the following Python libraries for this hackathon:


- `ipykernel`
- `jupyter`
- `matplotlib`
- `numpy`
- `pandas`
- `seaborn`
- `sklearn`


There are a multitude of ways to do this, but all involve setting up a virtual environment so you do not break Python across your computer. You can do this using [Python itself](https://docs.python.org/3/library/venv.html) or the [Anaconda package manager](https://www.anaconda.com/docs/getting-started/working-with-conda/environments), but my strong recommendation is to use [the uv venv manager](https://docs.astral.sh/uv/getting-started/installation/#__tabbed_1_2) to manage your python virtual environments, as it its the fastest, easiest, and the way I will best be able to help you.

 

To install uv on WINDOWS, open Powershell and type

```bat
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

On MAC, open bash and type:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

On LINUX, you can use the same method as Mac, or you can use your package manager, e.g.

```bash
sudo pacman -S uv
```

 


**ONCE YOU HAVE UV INSTALLED,** open a powershell/terminal in your cloned folder from github (probably something close to `~/Documents/GitHub/csci547_ML/` or `~/csci547_ML/`).

Then, use the command

```bash
uv init
```

to set up your environment, and, finally, run

```bash
uv add ipykernel jupyter matplotlib numpy pandas seaborn scikit-learn
```

wait a second, and your virtual environment should be complete! Open VSCode or Jupyter Notebooks and your new .venv named `csci547_ML` should automatically pop up! Adding and removing libraries is trivial (`uv add x` or `uv remove x`) and this Python installation will not poison your other ones.

---

## 1. Linear Regression with One Variable

In Hackathon 0, we used `scikit-learn` to fit a model to our data. Today, we will look "under the hood" and implement the mathematical engine that makes linear regression work from scratch. 

Suppose you are the CEO of a restaurant franchise and are considering different cities for opening a new food truck. You have historical data for profits and populations from existing cities, and you want to use this data to predict profit for potential expansion locations.

The file `ex1data1.txt` contains our dataset:
* Column 1: Population of a city (in 10,000s)
* Column 2: Profit of a food truck in that city (in $10,000s). *A negative value indicates a loss.*

### Think-Pair-Share 1: Feature Relationships

We are predicting food truck profit based on city population.

1. **Think** Mathematically and intuitively, what kind of relationship (linear, exponential, inversely proportional, etc.) do you expect to see between a city's population and a food truck's profit? Why?
 
    *Your individual hypothesis:* **ANSWER**

1. **Pair** Discuss your hypothesis with your group.

    *Group consensus:* **ANSWER**

1. **Share:** Be prepared to share your group's conclusion with the class.

---

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
try:

    from google.colab import files # Colab file import
    uploaded = files.upload()
    
    df = pd.read_csv('ex1data1.txt', sep=',', header=None)
    df.columns = ['population', 'profit']
    
except: 
    
    df = pd.read_csv('./data/linear_regression_hack/ex1data1.txt', sep=',', header=None) # local file import
    df.columns = ['population', 'profit']



In [ ]:
df.head()

In [ ]:
ax = sns.scatterplot(x='population', y='profit', data=df)
ax.set(xlabel='Population of City in 10,000s', ylabel='Profit in $10,000s', title='Scatter plot of training data');

The plot shows that they have a linear relationship.

### 1.2 Gradient Descent

We will fit the linear regression parameters (denoted as $\theta$) to our dataset using the gradient descent optimization algorithm. Let's map these mathematical formulations to the components of our model:

**1. The Hypothesis ($h_\theta(x)$):** 
This is our model's prediction. For a single variable, it defines a linear equation where $\theta_1$ is the weight (slope) and $\theta_0$ is the bias (y-intercept). 
$$ h_\theta(x) = \theta^Tx = \theta_0 + \theta_1x_1$$

**2. The Cost Function ($J(\theta)$):** 
This function quantifies the error of our model. We use the Mean Squared Error (MSE), which calculates the average squared difference between our predicted values ($h_\theta(x^{(i)})$) and the actual true values ($y^{(i)}$). Our objective is to minimize $J(\theta)$.
$$ J(\theta) = \frac{1}{2m} \sum_{i=1}^m (h_\theta(x^{(i)}) - y^{(i)})^2 $$

**3. The Update Rule:** 
To minimize $J(\theta)$, batch gradient descent iteratively updates the parameters by moving in the direction of the steepest descent (the most negative gradient). 
$$ \theta_j := \theta_j - \alpha \frac{1}{m} \sum_{i=1}^m (h_\theta(x^{(i)}) - y^{(i)})x_j^{(i)} $$

#### Implementation Detail
To vectorize our code and compute both $\theta_0$ and $\theta_1$ simultaneously using matrix multiplication, we add an extra column of `1`s to our input matrix `X`. This allows us to treat the bias $\theta_0$ as just another feature weight.

In [ ]:
m = df.shape[0]
X = np.hstack((np.ones((m,1)), df.population.values.reshape(-1,1)))
y = np.array(df.profit.values).reshape(-1,1)
theta = np.zeros(shape=(X.shape[1],1))

iterations = 1500
alpha = 0.01

#### 1.2.1 Computing the Cost $J(\theta)$

In [ ]:
def compute_cost_one_variable(X, y, theta):
    m = y.shape[0]
    h = X.dot(theta)
    J = (1/(2*m)) * (np.sum((h - y)**2))
    return J

In [ ]:
J = compute_cost_one_variable(X, y, theta)
print('With theta = [0 ; 0]\nCost computed =', J)
print('Expected cost value (approx) 32.07')

In [ ]:
J = compute_cost_one_variable(X, y, [[-1],[2]])
print('With theta = [-1 ; 2]\nCost computed =', J)
print('Expected cost value (approx) 54.24')

#### 1.2.4 The Gradient Descent Algorithm
Gradient descent is a generic optimization algorithm that measures the local gradient of the cost function with regards to the parameter $\theta$ and steps in the direction of the descending gradient.

We repeat this update until convergence:
$$\theta_j := \theta_j - \alpha \frac{\partial}{\partial\theta_j}J(\theta_0, \theta_1) = \theta_j - \alpha \frac{1}{m} \sum_{i=1}^m (h_\theta(x^{(i)}) - y^{(i)})x_j^{(i)} $$

Here, $\alpha$ (alpha) is the **learning rate**, which controls the step size during optimization.

**Pause point:** Consider the role of the learning rate $\alpha$. What happens to the optimization process if $\alpha$ is too small? Conversely, what are the mathematical risks if $\alpha$ is too large?

### Think-Pair-Share 2: The Learning Rate ($\alpha$)

We are about to run Gradient Descent. Our update rule relies on $\alpha$ (the learning rate) to control the step size during optimization.

1. **Think:** Consider the role of the learning rate $\alpha$. What happens to the optimization process if $\alpha$ is infinitesimally small? Conversely, what are the mathematical risks to our cost function $J(\theta)$ if $\alpha$ is too large?

    *Your individual hypothesis:* **ANSWER**

1. **Pair (2 minutes):** Discuss your hypotheses with your group. Did you agree on the convergence behavior?

    *Group consensus:* **ANSWER**

1. **Share:** Be prepared to share your group's conclusion with the class.

---

In [ ]:
def gradient_descent(X, y, theta, alpha, num_iters):
    m = y.shape[0]
    J_history = np.zeros(shape=(num_iters, 1))

    for i in range(0, num_iters):
        h = X.dot(theta)
        diff_hy = h - y

        delta = (1/m) * (diff_hy.T.dot(X))
        theta = theta - (alpha * delta.T)
        J_history[i] = compute_cost_one_variable(X, y, theta)

    return theta, J_history

In [ ]:
theta, _ = gradient_descent(X, y, theta, alpha, iterations)
print('Theta found by gradient descent:\n', theta)
print('Expected theta values (approx)\n -3.6303\n  1.1664')

#### Plot the linear fit:

In [ ]:
ax = sns.scatterplot(x='population', y='profit', data=df)
plt.plot(X[:,1], X.dot(theta), color='r')
ax.set(xlabel='Population of City in 10,000s', ylabel='Profit in $10,000s', title='Training data with linear regression fit');

In [ ]:
y_pred = np.array([1, 3.5]).dot(theta)
f'For population = 35,000, we predict a profit of {y_pred[0]*10000}'

In [ ]:
y_pred = np.array([1, 7]).dot(theta)
f'For population = 70,000, we predict a profit of {y_pred[0]*10000}'

### 1.3 Visualizing $J(\theta)$

The cost function $J(\theta)$ is bowl-shaped and has a global mininum. This minimum is the optimal point for $\theta_0$ and $\theta_1$, and each step of gradient descent moves closer to this point.

In [ ]:
theta0_vals = np.linspace(-10, 10, 100)
theta1_vals = np.linspace(-1, 4, 100)

In [ ]:
J_vals = np.zeros(shape=(len(theta0_vals), len(theta1_vals)))

In [ ]:
for i in range(0, len(theta0_vals)):
    for j in range(0, len(theta1_vals)):
        J_vals[i,j] = compute_cost_one_variable(X, y, [[theta0_vals[i]], [theta1_vals[j]]])

In [ ]:
ax = plt.contour(theta0_vals, theta1_vals, np.transpose(J_vals), levels=np.logspace(-2,3,20))
plt.plot(theta[0,0], theta[1,0], marker='x', color='r');
plt.xlabel(r'$\theta_0$');
plt.ylabel(r'$\theta_1$');
plt.title('Contour, showing minimum');

### 1.4 Equivalent Code using Scikit-Learn

Implementing gradient descent from scratch provides valuable insight into the underlying optimization process. However, in practice, we rely on optimized, production-ready libraries like `scikit-learn` to handle these computations efficiently.

Notice how the `LinearRegression` class abstracts away the cost function, learning rate, and iteration loop, achieving the same result in just three lines of code.

In [ ]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(df.population.values.reshape(-1,1),
            df.profit.values.reshape(-1,1))

In [ ]:
lin_reg.intercept_, lin_reg.coef_

---

## 2. Linear Regression with Multiple Variables

Now we will extend our model to handle multiple input features (multivariate linear regression). Because we vectorized our cost function and gradient descent algorithms in Part 1 using matrix multiplication, the underlying optimization code will remain largely the same. However, adding more dimensions introduces new challenges in how we prepare our data.

**The Scenario:** Suppose you are building a model to predict housing prices in Portland, Oregon, based on historical sales data to determine the market value of new properties.

The file `ex1data2.txt` contains our training set:
* Column 1: Size of the house (in square feet)
* Column 2: Number of bedrooms
* Column 3: Price of the house (Our target variable $y$)

### 2.1 Feature Normalization

In [ ]:
try:

    from google.colab import files # Colab file import
    uploaded = files.upload()
    
    df2 = pd.read_csv('./data/linear_regression_hack/ex1data2.txt', sep=',', header=None)
    df2.columns = ['house_size', 'bedrooms', 'house_price']
    
except: 
    
    df2 = pd.read_csv('./data/linear_regression_hack/ex1data2.txt', sep=',', header=None) # local file import
    df2.columns = ['house_size', 'bedrooms', 'house_price']



In [ ]:
df2.describe().T

**Pause point:** Look at the summary statistics above. House sizes are in the 1000s, but the number of bedrooms is generally between 1 and 5. From a mathematical perspective, why might this disparity in scale be problematic for the gradient descent algorithm?

### Think-Pair-Share 3: Feature Dominance
**Context:** Look at the summary statistics above. House sizes are in the 1000s (e.g., 1600 sq ft), but the number of bedrooms is generally between 1 and 5. 

1. **Think:** From a mathematical perspective, why might this disparity in scale be problematic for the gradient descent algorithm computing the cost function $J(\theta)$?

    *Your individual hypothesis:* **ANSWER**

2. **Pair (2 minutes):** Discuss with your group how the algorithm might incorrectly assign "importance" to weights based strictly on raw magnitude.

    *Group consensus:* **ANSWER**

**3. Share:** Be prepared to share your group's conclusion with the class.

---

When features differ by orders of magnitude, features with larger ranges can disproportionately dominate the cost function, causing gradient descent to converge very slowly or oscillate. 

We resolve this by bringing all features to a similar scale (ideally $-1 \leq x_i \leq 1$). We will use **Feature Normalization**:
$$x_i := \frac{x_i - \mu_i}{s_i}$$
*(Where $\mu_i$ is the mean of the feature, and $s_i$ is the standard deviation).*

In [ ]:
def feature_normalize(X, mean=np.zeros(1), std=np.zeros(1)):
    X = np.array(X)
    if len(mean.shape) == 1 or len(std.shape) == 1:
        mean = np.mean(X, axis=0)
        std = np.std(X, axis=0, ddof=1)

    X = (X - mean)/std
    return X, mean, std

In [ ]:
X_norm, mu, sigma = feature_normalize(df2[['house_size', 'bedrooms']])

In [ ]:
df2['house_size_normalized'] = X_norm[:,0]
df2['bedrooms_normalized'] = X_norm[:,1]
df2[['house_size_normalized', 'bedrooms_normalized']].describe().T

### 2.2 Gradient Descent

The only difference from univariate regression problem is that now there is one more feature in the matrix X. The hypothesis function and the batch gradient descent update rule remain unchanged.

Note: In the multivariate case, the cost function can also be written in the following vectorized form:

$$J(\theta) = \frac{1}{2m}(X\theta-y)^T(X\theta-y)$$

In [ ]:
def compute_cost(X, y, theta):
    m = y.shape[0]
    h = X.dot(theta)
    J = (1/(2*m)) * ((h-y).T.dot(h-y))
    return J

In [ ]:
def gradient_descent(X, y, theta, alpha, num_iters):
    m = y.shape[0]
    J_history = np.zeros(shape=(num_iters, 1))

    for i in range(0, num_iters):
        h = X.dot(theta)
        diff_hy = h - y

        delta = (1/m) * (diff_hy.T.dot(X))
        theta = theta - (alpha * delta.T)
        J_history[i] = compute_cost(X, y, theta)

    return theta, J_history

#### 2.2.1 Selecting Learning Rates

Tips:
* Make a plot with number of iterations on the x-axis. Now plot the cost function, $J(\theta)$ over the number of iterations of gradient descent. If $J(\theta)$  ever increases, then you probably need to decrease $\alpha$.
* Declare convergence if $J(\theta)$ decreases by less than E in one iteration, where E is some small value such as $10^{−3}$.

In [ ]:
m = df2.shape[0]
X = np.hstack((np.ones((m,1)),X_norm))
y = np.array(df2.house_price.values).reshape(-1,1)
theta = np.zeros(shape=(X.shape[1],1))

In [ ]:
alpha = [0.3, 0.1, 0.03, 0.01]
colors = ['b','r','g','c']
num_iters = 50

In [ ]:
for i in range(0, len(alpha)):
    theta = np.zeros(shape=(X.shape[1],1))
    theta, J_history = gradient_descent(X, y, theta, alpha[i], num_iters)
    plt.plot(range(len(J_history)), J_history, colors[i], label='Alpha {}'.format(alpha[i]))
plt.xlabel('Number of iterations');
plt.ylabel('Cost J');
plt.title('Selecting learning rates');
plt.legend()
plt.show()

In [ ]:
iterations = 250
alpha = 0.1
theta, _ = gradient_descent(X, y, theta, alpha, iterations)

print('Theta found by gradient descent:')
print(theta)

##### Estimate the price of a 1650 sq-ft, 3 bedrooms house

In [ ]:
sqft = (1650 - mu[0])/sigma[0]
bedrooms = (3 - mu[1])/sigma[1]
y_pred = theta[0] + theta[1]*sqft + theta[2]*bedrooms
f'Price of a house with 1650 square feet and 3 bedrooms: {y_pred[0]}$'

### 2.3 Normal Equations

A closed-form solution to find $\theta$ without iteration.

$$\theta = (X^TX)^{-1}X^Ty$$

In [ ]:
def normal_eqn(X, y):
    inv = np.linalg.pinv(X.T.dot(X))
    theta = inv.dot(X.T).dot(y)
    return theta

In [ ]:
Xe = np.hstack((np.ones((m,1)),df2[['house_size', 'bedrooms']].values))
theta_e = normal_eqn(Xe, y)
theta_e

In [ ]:
y_pred = theta_e[0] + theta_e[1]*1650 + theta_e[2]*3
f'Price of a house with 1650 square feet and 3 bedrooms: {y_pred[0]}$'

### 2.4 Implementation using Scikit-Learn

Implementing gradient descent and the normal equation from scratch is crucial for understanding the mathematical foundation of machine learning models. However, in a production environment, we rely on highly optimized libraries.

Let's revisit `scikit-learn` (from Hackathon 0). Notice how the library abstracts away the gradient descent loop and the feature padding, resolving the regression problem in just a few lines of code. (Note: Under the hood, `LinearRegression` in sklearn actually computes the least squares solution using Singular Value Decomposition (SVD), making it highly robust!).

In [ ]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(X_norm, y)

In [ ]:
lin_reg.intercept_, lin_reg.coef_

---

## Hackathon Takeaways 

As your group finishes this notebook, collaborate to answer the following synthesis questions. Retrieving and contrasting this information is crucial for cementing your understanding!

1. **Trade-off Analysis:** 
In Part 1, we used iterative Gradient Descent. In Part 2, we introduced the closed-form Normal Equation. What is one mathematical or computational advantage of using Gradient Descent over the Normal Equation when dealing with massive datasets (e.g., millions of features)?

    **ANSWER**

2. **Diagnosing Failures:** 
If a future machine learning model's Cost Function $J(\theta)$ is strictly *increasing* after every epoch instead of decreasing, what is the very first hyperparameter you would check based on what you learned today?

    **ANSWER**

3. **3-2-1 Summary:**
* **3** key mathematical or programming concepts your group solidified today:

    1. 

    2. 

    3. 

* **2** things you are still slightly confused about (we will address these in the next lecture!):

    1. 

    2. 

* **1** real-world scenario where you would apply Multivariate Linear Regression:
    
    1.